# Greedy Optimisation Prototype

> Mario Boley

Spike to explore implementation of [GitHub Issue 3](https://github.com/marioboley/optikon/issues/3).

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
import numpy as np

from testdata import mvn_with_correlation
x = mvn_with_correlation(100, seed=0)
y = np.random.default_rng(seed=0).normal(size=100)

In [12]:
import numpy as np
from optikon import Propositionalization
from numba import njit

@njit
def argsort_columns(x):
    n, p = x.shape
    out = np.empty((n, p), dtype=np.int64)
    for j in range(p):
        out[:, j] = np.argsort(x[:, j])
    return out

@njit
def max_weighted_support_greedy(x, y, max_depth=5):
    n, p = x.shape
    orders = argsort_columns(x)
    support = np.ones(n, dtype=np.bool)
    support_count = n

    v = np.zeros(max_depth, dtype=np.int64)
    s = np.zeros(max_depth, dtype=np.int64)
    t = np.zeros(max_depth, dtype=np.float64)

    current = np.zeros(p, dtype=np.int64) # cursor buffer for order updates
    
    best_sum = np.sum(y)
    num_cond = 0

    for k in range(1, max_depth+1):
        sum_y = np.sum(y[orders[:support_count, 0]])
        best_j, best_i, best_s = -1, -1, 1
        improvement = False
        for j in range(p):
            sum_left, sum_right = 0, sum_y
            for i in range(support_count - 1): 
                # test splits between x^j_i (last left) and x^j_i+1 (first right)
                y_i = y[orders[i, j]]
                sum_left += y_i
                sum_right -= y_i
                if sum_left > best_sum:
                    best_i = i
                    best_j = j
                    best_s = -1
                    best_sum = sum_left
                    improvement = True
                elif sum_right > best_sum:
                    best_i = i
                    best_j = j
                    best_s = 1
                    best_sum = sum_right
                    improvement = True

        if not improvement:
            break

        v[k-1] = best_j
        s[k-1] = best_s
        t[k-1] = (x[orders[best_i, best_j], best_j] + x[orders[best_i + 1, best_j], best_j]) / 2
        num_cond = k

        if best_s == 1: # lower bound
            support[orders[:best_i+1, best_j]] = False
            #orders[:support_count, best_j] = orders[best_i+1:best_i+1+support_count, best_j]
        else: # upper bound
            support[orders[best_i+1:, best_j]] = False

        current[:] = 0
        for i in range(support_count): # need old support count here
            for j in range(p): # can this loop be vectorised?
                if support[orders[i, j]]:
                    orders[current[j], j] = orders[i, j]
                    current[j] += 1

        if best_s == 1: # lower bound
            support_count = support_count - best_i - 1
        else: # upper bound
            support_count = best_i + 1
    res = Propositionalization(v[:num_cond], t[:num_cond], s[:num_cond])
    return res, best_sum

res, val = max_weighted_support_greedy(x, y)
str(res), val

('[x4 <= 0.494, x1 <= -1.514, x2 >= 0.140]', 21.443390305350473)

In [6]:
%timeit max_weighted_support_greedy(x, y)

5.18 μs ± 114 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [7]:
%timeit max_weighted_support_greedy.py_func(x, y)

363 μs ± 589 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [8]:
from optikon import max_weighted_support, equal_width_propositionalization

max_weighted_support(x, y, equal_width_propositionalization(x))

(array([ 8, 16, 59, 66]), 20.105766513335315, 5244, 22996)

In [9]:
%timeit max_weighted_support(x, y, equal_width_propositionalization(x))

45.9 ms ± 346 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [11]:
from testdata import SMALL_1

res, val = max_weighted_support_greedy(SMALL_1.x, SMALL_1.y)
str(res), val

('[x1 >= 0.500, x3 >= 0.500]', 3)